# UniMol v2 — PXR pEC50 Fine-tuning (20-Fold, Kaggle)

**Pipeline**
1. Generate 3D conformers with RDKit ETKDG (on Kaggle hardware for consistency)
2. Fine-tune UniMol v2 (84 M) — 20-fold scaffold CV
3. OOF evaluation + isotonic recalibration
4. Test set predictions → submission CSV

**Setup:** Add your dataset containing `clean_train.csv` and `test.csv` via *Add Data*.

## Cell 1 — GPU Check

In [ ]:
import subprocess, sys
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    raise SystemExit('No GPU detected — enable GPU accelerator in Kaggle settings.')

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
print(f"GPU total: {torch.cuda.mem_get_info()[1]/1e9:.2f} GB")

## Cell 2 — Install Dependencies

In [ ]:
!pip install -q unimol-tools huggingface_hub

from unimol_tools import MolTrain, MolPredict
from rdkit import Chem
print('unimol-tools + rdkit ready')

In [ ]:
"""
train_unimol_10fold_kaggle.py
=============================
8-fold scaffold CV fine-tuning of Uni-Mol v2 (84m) on PXR pEC50 data.
3D conformers are generated ON Kaggle hardware (ETKDG v3 + MMFF94) so
geometry is deterministic and not hardware-dependent.
Includes OOF collection and isotonic recalibration.

Design
------
• 3D conformers generated before training (ETKDG v3 + MMFF94, fixed seed).
• ONE MolTrain call with split='scaffold', kfold=20 — this is exactly how
  the original working script trains, proven not to raise CheckpointError or
  OOM errors.  unimol_tools handles the 20-fold split internally and saves
  model_0.pth ... model_19.pth in the save directory.
• Test predictions via MolPredict which auto-ensembles all 20 models.
• OOF via scaffold split replication + per-fold temp dirs (same approach as
  the working Colab notebook Cell-7 fix).
• Isotonic recalibration on OOF predictions (optional, on by default).

Usage on Kaggle
---------------
    !python train_unimol_10fold_kaggle.py
or, as a notebook cell:
    from train_unimol_20fold_kaggle import main; main()
"""

import os
import sys
import re
import time
import gc
import glob
import shutil
import tempfile
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.isotonic import IsotonicRegression

warnings.filterwarnings("ignore")

# ==============================================================================
# MONKEY-PATCH: Fix torch.utils.checkpoint for PyTorch 2.x
# ==============================================================================
try:
    import torch.utils.checkpoint as _ckpt
    _orig_ckpt = _ckpt.checkpoint

    def _patched_ckpt(function, *args, **kwargs):
        kwargs.setdefault('use_reentrant', False)
        return _orig_ckpt(function, *args, **kwargs)

    _ckpt.checkpoint = _patched_ckpt
    print("[init] CheckpointError patch applied.")
except Exception as _e:
    print(f"[init] CheckpointError patch skipped: {_e}")

# ==============================================================================
# ENVIRONMENT DETECTION
# ==============================================================================
IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = 'google.colab' in sys.modules

if IS_KAGGLE:
    DATA_DIR = '/kaggle/input/datasets/gashawmg/pxr-newdatasets'  # update if needed
    WORK_DIR = '/kaggle/working'
elif IS_COLAB:
    DATA_DIR = '/content/drive/MyDrive/unimol'
    WORK_DIR = '/content/drive/MyDrive/unimol'
else:
    DATA_DIR = '.'
    WORK_DIR = '.'

# ==============================================================================
# CONFIGURATION
# ==============================================================================
N_FOLDS    = 8
EPOCHS     = 50
PATIENCE   = 15

# File names (searched in DATA_DIR then WORK_DIR recursively)
TRAIN_CSV_NAME = 'clean_train2.csv'   # all 3729 labeled compounds
TEST_CSV_NAME  = 'test.csv'

KFOLD_SAVE_DIR = os.path.join(WORK_DIR, 'unimol_kfold_7')
OUTPUT_CSV     = os.path.join(WORK_DIR, 'predictions_unimol_7fold.csv')

# Set COMPUTE_OOF=True to run OOF reconstruction and isotonic calibration.
# OOF requires scaffold split replication which may not perfectly match
# unimol_tools' internal split, but is close enough for calibration.
COMPUTE_OOF       = True
APPLY_CALIBRATION = True    # only used when COMPUTE_OOF=True

# GPU / memory settings
if IS_COLAB or IS_KAGGLE:
    BATCH_SIZE = 4
    REMOVE_HS  = False   # strongly recommended for 20-fold on T4 / P100
# else:
#     BATCH_SIZE = 1
#     REMOVE_HS  = True

LEARNING_RATE = 1e-4

# Column names
SMILES_TRAIN = 'smiles'    # column in clean_train.csv (normalised below)
SMILES_TEST  = 'SMILES'    # column in test.csv
TARGET_COL   = 'pEC50'

# Conformer generation settings
CONFORMER_SEED     = 42
CONFORMER_ATTEMPTS = 5

# ==============================================================================
# FILE DISCOVERY
# ==============================================================================
def _find_file(name):
    for base in [DATA_DIR, WORK_DIR]:
        p = os.path.join(base, name)
        if os.path.exists(p):
            return p
        if os.path.isdir(base):
            for root, _dirs, files in os.walk(base):
                if name in files:
                    return os.path.join(root, name)
    return os.path.join(DATA_DIR, name)   # will fail clearly

TRAIN_CSV_PATH = _find_file(TRAIN_CSV_NAME)
TEST_CSV_PATH  = _find_file(TEST_CSV_NAME)

# ==============================================================================
# HELPERS
# ==============================================================================
def flush_gpu():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
    except Exception:
        pass


def gpu_info():
    try:
        import torch
        if torch.cuda.is_available():
            free_gb  = torch.cuda.mem_get_info()[0] / 1e9
            total_gb = torch.cuda.mem_get_info()[1] / 1e9
            print(f"  GPU : {torch.cuda.get_device_name(0)}")
            print(f"  VRAM: {free_gb:.1f} GB free / {total_gb:.1f} GB total")
        else:
            print("  WARNING: No CUDA GPU detected — training will be very slow")
    except Exception as e:
        print(f"  GPU info: {e}")


def prevent_sleep():
    if IS_COLAB or IS_KAGGLE:
        return
    try:
        import ctypes
        ctypes.windll.kernel32.SetThreadExecutionState(0x80000003)
        print("  Sleep prevention: ACTIVE")
    except Exception:
        pass


def restore_sleep():
    if IS_COLAB or IS_KAGGLE:
        return
    try:
        import ctypes
        ctypes.windll.kernel32.SetThreadExecutionState(0x80000000)
    except Exception:
        pass


def verify_saved_models(save_dir):
    models = sorted(
        glob.glob(os.path.join(save_dir, '*.pth')) +
        glob.glob(os.path.join(save_dir, '**', '*.pth'), recursive=True))
    print(f"  Saved model files: {len(models)}")
    for m in models[:6]:
        print(f"    {os.path.basename(m)}  ({os.path.getsize(m)/1e6:.0f} MB)")


# ==============================================================================
# 3-D CONFORMER GENERATION  (ETKDG v3 + MMFF94)
# ==============================================================================
def generate_conformers(smiles_list, remove_hs=REMOVE_HS,
                        max_attempts=CONFORMER_ATTEMPTS, label=""):
    """
    Generate one 3-D conformer per molecule using RDKit ETKDG v3 + MMFF94.

    Returns
    -------
    atoms_list  : list[list[str] | None]
    coords_list : list[list[list[float]] | None]
    valid_mask  : list[bool]
    """
    from rdkit import Chem
    from rdkit.Chem import AllChem

    atoms_list  = []
    coords_list = []
    valid_mask  = []
    n_failed    = 0
    tag = f"[{label}] " if label else ""

    for idx, smi in enumerate(smiles_list):
        if idx % 250 == 0:
            print(f"    {tag}Conformer {idx}/{len(smiles_list)} ...")

        mol_out = None
        for attempt in range(max_attempts):
            try:
                mol = Chem.MolFromSmiles(smi)
                if mol is None:
                    break
                mol = Chem.AddHs(mol)
                params = AllChem.ETKDGv3()
                params.randomSeed = CONFORMER_SEED + attempt
                if AllChem.EmbedMolecule(mol, params) == -1:
                    continue
                res = AllChem.MMFFOptimizeMolecule(mol, maxIters=2000)
                if res == -1:
                    continue
                if remove_hs:
                    mol = Chem.RemoveHs(mol)
                mol_out = mol
                break
            except Exception:
                continue

        if mol_out is not None:
            conf   = mol_out.GetConformer()
            atoms  = [atom.GetSymbol() for atom in mol_out.GetAtoms()]
            coords = conf.GetPositions().tolist()
            atoms_list.append(atoms)
            coords_list.append(coords)
            valid_mask.append(True)
        else:
            atoms_list.append(None)
            coords_list.append(None)
            valid_mask.append(False)
            n_failed += 1

    n_ok = sum(valid_mask)
    print(f"    {tag}Done: {n_ok}/{len(smiles_list)} OK  ({n_failed} failed)")
    return atoms_list, coords_list, valid_mask


# ==============================================================================
# SCAFFOLD SPLIT (for OOF reconstruction)
# ==============================================================================
def scaffold_kfold_split(smiles_list, n_folds, seed=42):
    """
    Balanced Murcko-scaffold k-fold split (greedy, deterministic).
    Attempts to match unimol_tools' internal split for OOF reconstruction.

    Returns
    -------
    fold_val_indices : list[list[int]]
        fold_val_indices[i] = sorted list of val indices for fold i
    """
    from rdkit import Chem
    from rdkit.Chem.Scaffolds import MurckoScaffold

    scaffold_to_indices = defaultdict(list)
    for i, smi in enumerate(smiles_list):
        try:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                sc = f'__INVALID_{i}__'
            else:
                sc_mol = MurckoScaffold.GetScaffoldForMol(mol)
                sc     = Chem.MolToSmiles(sc_mol)
        except Exception:
            sc = f'__ERROR_{i}__'
        scaffold_to_indices[sc].append(i)

    # Deterministic order: largest groups first, tie-break by first index
    scaffold_sets = sorted(
        scaffold_to_indices.values(),
        key=lambda grp: (-len(grp), grp[0])
    )

    fold_val_indices = [[] for _ in range(n_folds)]
    fold_sizes       = [0] * n_folds
    for group in scaffold_sets:
        min_fold = int(np.argmin(fold_sizes))
        fold_val_indices[min_fold].extend(group)
        fold_sizes[min_fold] += len(group)

    sizes = [len(f) for f in fold_val_indices]
    print(f"  Scaffold val-fold sizes: "
          f"min={min(sizes)}, max={max(sizes)}, mean={np.mean(sizes):.1f}")
    return fold_val_indices


# ==============================================================================
# TRAINING
# ==============================================================================
def run_training(train_smiles, train_atoms, train_coords, train_targets,
                 save_dir):
    """
    Train all N_FOLDS models in ONE MolTrain call with split='scaffold'.
    Conformers are passed as a dict; unimol_tools does the scaffold splitting
    internally and saves model_0.pth … model_{N_FOLDS-1}.pth.
    """
    from unimol_tools import MolTrain

    os.makedirs(save_dir, exist_ok=True)
    flush_gpu()

    # Build input data dict with pre-computed conformers
    # (no 'split' key — scaffold split is handled by MolTrain internally)
    use_conf = all(a is not None for a in train_atoms)
    if use_conf:
        data = {
            'SMILES':      train_smiles,
            'atoms':       train_atoms,
            'coordinates': train_coords,
            'TARGET':      train_targets,
        }
        print(f"  Using pre-computed 3D conformers for all "
              f"{len(train_smiles):,} molecules.")
    else:
        n_missing = sum(1 for a in train_atoms if a is None)
        print(f"  WARNING: {n_missing} conformers missing; "
              "falling back to SMILES-only (unimol_tools generates conformers).")
        data = pd.DataFrame({
            'SMILES':  train_smiles,
            'TARGET':  list(train_targets),
        })

    sep = "-" * 65
    print(f"\n{sep}")
    print(f"  split=scaffold  kfold={N_FOLDS}  epochs={EPOCHS}  "
          f"patience={PATIENCE}")
    print(f"  batch_size={BATCH_SIZE}  remove_hs={REMOVE_HS}  "
          f"lr={LEARNING_RATE}")
    print(f"  save_path={save_dir}")
    print(sep)

    clf = MolTrain(
        task           = 'regression',
        data_type      = 'molecule',
        epochs         = EPOCHS,
        learning_rate  = LEARNING_RATE,
        batch_size     = BATCH_SIZE,
        early_stopping = PATIENCE,
        metrics        = 'mae',
        split          = 'scaffold',
        kfold          = N_FOLDS,
        save_path      = save_dir,
        remove_hs      = REMOVE_HS,
        smiles_col     = 'SMILES',
        target_cols    = 'TARGET',
        model_name     = 'unimolv2',
        model_size     = '84m',
        use_cuda       = True,
    )
    clf.fit(data=data)
    return clf


# ==============================================================================
# INFERENCE
# ==============================================================================
def run_inference(save_dir, smiles_list, atoms_list=None, coords_list=None):
    """
    Load a saved model directory and predict.  MolPredict automatically
    ensembles all model_*.pth files found in save_dir.

    Passes pre-computed conformers when all are valid; otherwise uses SMILES.
    Returns np.ndarray of shape (N,).
    """
    from unimol_tools import MolPredict

    predictor = MolPredict(load_model=save_dir)

    use_conf = (atoms_list is not None
                and all(a is not None for a in atoms_list))

    if use_conf:
        data = {
            'SMILES':      smiles_list,
            'atoms':       atoms_list,
            'coordinates': coords_list,
        }
    else:
        data = pd.DataFrame({'SMILES': smiles_list})

    preds = predictor.predict(data=data)

    if isinstance(preds, np.ndarray):
        return preds.flatten()
    if isinstance(preds, pd.DataFrame):
        return preds.iloc[:, 0].values.flatten()
    if isinstance(preds, list):
        return np.array(preds).flatten()
    return np.array(preds).flatten()


# ==============================================================================
# OOF RECONSTRUCTION  (per-fold temp dir approach)
# ==============================================================================
def compute_oof(save_dir, train_smiles, train_targets,
                train_atoms, train_coords):
    """
    Reconstruct OOF predictions by:
      1. Replicating the scaffold split to identify each fold's val set.
      2. For each fold, creating a temp dir with that fold's model_N.pth
         and a config.yaml patched to kfold=1 so MolPredict loads one model.
      3. Predicting on that fold's val molecules.

    Returns oof_preds : np.ndarray of shape (N_train,)
    """
    # ── Step 1: replicate scaffold split ─────────────────────────────────────
    print(f"  Replicating {N_FOLDS}-fold scaffold split for OOF ...")
    fold_val_indices = scaffold_kfold_split(
        train_smiles, n_folds=N_FOLDS, seed=CONFORMER_SEED)

    # ── Step 2: find saved models and config ──────────────────────────────────
    model_files = sorted(
        glob.glob(os.path.join(save_dir, 'model_*.pth')))
    print(f"  Found {len(model_files)} model files in {save_dir}")

    config_src = os.path.join(save_dir, 'config.yaml')
    scaler_src = os.path.join(save_dir, 'target_scaler.ss')
    if not os.path.exists(config_src):
        print("  WARNING: config.yaml not found — OOF skipped.")
        return None

    # Read config template once
    with open(config_src, 'r') as f:
        config_text = f.read()

    # Patch that reduces any kfold/n_model counter to 1
    def patch_config(text):
        for key in ('kfold', 'fold_num', 'n_model', 'num_fold'):
            text = re.sub(
                rf'({key}\s*:\s*)\d+',
                r'\g<1>1',
                text)
        return text

    config_patched = patch_config(config_text)

    # ── Step 3: predict per fold ───────────────────────────────────────────────
    oof_preds = np.zeros(len(train_smiles))

    for fold_idx, val_idx in enumerate(fold_val_indices):
        if fold_idx >= len(model_files):
            print(f"  WARNING: no model file for fold {fold_idx} — skipping")
            continue

        va_smiles = [train_smiles[i] for i in val_idx]
        va_atoms  = [train_atoms[i]  for i in val_idx]
        va_coords = [train_coords[i] for i in val_idx]
        va_true   = [train_targets[i] for i in val_idx]

        with tempfile.TemporaryDirectory() as tmp_dir:
            # Copy this fold's model as model_0.pth
            shutil.copy(model_files[fold_idx],
                        os.path.join(tmp_dir, 'model_0.pth'))

            # Write patched config
            with open(os.path.join(tmp_dir, 'config.yaml'), 'w') as f:
                f.write(config_patched)

            # Copy scaler if available
            if os.path.exists(scaler_src):
                shutil.copy(scaler_src,
                            os.path.join(tmp_dir, 'target_scaler.ss'))

            # Predict
            try:
                fold_preds = run_inference(
                    tmp_dir, va_smiles,
                    va_atoms  if all(a is not None for a in va_atoms)  else None,
                    va_coords if all(c is not None for c in va_coords) else None,
                )
            except Exception as e:
                print(f"  WARNING: OOF inference failed for fold {fold_idx}: {e}")
                continue

        fold_mae      = np.mean(np.abs(fold_preds - np.array(va_true)))
        fold_spearman = spearmanr(fold_preds, va_true).correlation
        print(f"  Fold {fold_idx+1:2d}/{N_FOLDS}  "
              f"val_n={len(val_idx):4d}  "
              f"MAE={fold_mae:.4f}  Spearman={fold_spearman:.4f}")

        for idx, pred in zip(val_idx, fold_preds):
            oof_preds[idx] = pred

    return oof_preds


# ==============================================================================
# SUBMISSION HELPER
# ==============================================================================
def save_predictions(test_df, pred_values, output_csv):
    pred_values = np.array(pred_values).flatten()
    if len(pred_values) != len(test_df):
        print(f"  WARNING: {len(pred_values)} preds for {len(test_df)} rows — truncating")
        pred_values = pred_values[:len(test_df)]

    name_col = next(
        (c for c in test_df.columns
         if 'molecule' in c.lower() and 'name' in c.lower()), None)

    sub = pd.DataFrame()
    sub['Molecule Name'] = (test_df[name_col].values if name_col
                            else [f"compound_{i+1}" for i in range(len(test_df))])
    sub['SMILES'] = test_df[SMILES_TEST].values
    sub['pEC50']  = pred_values

    sub.to_csv(output_csv, index=False)
    print(f"  Saved → {output_csv}")
    print(f"  N={len(sub):,}  "
          f"range=[{pred_values.min():.4f}, {pred_values.max():.4f}]  "
          f"mean={pred_values.mean():.4f}  std={pred_values.std():.4f}")
    return sub


# ==============================================================================
# MAIN
# ==============================================================================
def main():
    sep = "=" * 65
    t0  = time.time()

    print(sep)
    env_label = "Kaggle" if IS_KAGGLE else ("Colab" if IS_COLAB else "Local")
    print(f"Uni-Mol {N_FOLDS}-Fold Scaffold CV  [{env_label}]")
    print(f"  Epochs       : {EPOCHS}  Patience: {PATIENCE}")
    print(f"  Batch size   : {BATCH_SIZE}  LR: {LEARNING_RATE}")
    print(f"  Remove HS    : {REMOVE_HS}")
    print(f"  OOF          : {COMPUTE_OOF}  Calibration: {APPLY_CALIBRATION}")
    print(f"  Train CSV    : {TRAIN_CSV_PATH}")
    print(f"  Test CSV     : {TEST_CSV_PATH}")
    print(f"  Save dir     : {KFOLD_SAVE_DIR}")
    print(f"  Output CSV   : {OUTPUT_CSV}")
    print(sep)

    prevent_sleep()

    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
        "expandable_segments:True,max_split_size_mb:64")
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    os.environ["CUDA_VISIBLE_DEVICES"]   = "0"

    flush_gpu()
    gpu_info()

    # ── Install unimol_tools if needed ────────────────────────────────────────
    try:
        from unimol_tools import MolTrain, MolPredict
        print("  unimol_tools: OK")
    except ImportError:
        if IS_KAGGLE or IS_COLAB:
            print("  Installing unimol_tools ...")
            import subprocess
            subprocess.check_call(
                [sys.executable, '-m', 'pip', 'install', 'unimol_tools', '-q'])
            from unimol_tools import MolTrain, MolPredict
            print("  unimol_tools installed.")
        else:
            raise

    # ── Load data ─────────────────────────────────────────────────────────────
    print(f"\nLoading training data: {TRAIN_CSV_PATH}")
    train_df = pd.read_csv(TRAIN_CSV_PATH)
    train_df.columns = [c.strip() for c in train_df.columns]

    # Normalise SMILES column name
    smiles_actual = next(
        (c for c in train_df.columns if c.lower() == 'smiles'), SMILES_TRAIN)
    if smiles_actual != 'SMILES':
        train_df = train_df.rename(columns={smiles_actual: 'SMILES'})

    train_df = train_df.dropna(
        subset=['SMILES', TARGET_COL]).reset_index(drop=True)
    print(f"  {len(train_df):,} training compounds")
    print(f"  pEC50: [{train_df[TARGET_COL].min():.2f}, "
          f"{train_df[TARGET_COL].max():.2f}]  "
          f"mean={train_df[TARGET_COL].mean():.3f}")

    print(f"\nLoading test data: {TEST_CSV_PATH}")
    test_df = pd.read_csv(TEST_CSV_PATH)
    test_df.columns = [c.strip() for c in test_df.columns]
    test_df = test_df.dropna(subset=[SMILES_TEST]).reset_index(drop=True)
    print(f"  {len(test_df):,} test compounds")

    # ── Generate 3-D conformers ───────────────────────────────────────────────
    print(f"\n[Step 1/4] Generating training conformers "
          f"({len(train_df)} molecules) ...")
    t_conf = time.time()
    train_atoms, train_coords, train_valid = generate_conformers(
        train_df['SMILES'].tolist(), remove_hs=REMOVE_HS, label="train")
    print(f"  Training conformers done in {(time.time()-t_conf):.0f} s")

    # Drop molecules where conformer generation failed entirely
    failed = [i for i, v in enumerate(train_valid) if not v]
    if failed:
        print(f"  Dropping {len(failed)} molecules with failed conformers")
        keep         = [i for i in range(len(train_df)) if train_valid[i]]
        train_df     = train_df.iloc[keep].reset_index(drop=True)
        train_atoms  = [train_atoms[i]  for i in keep]
        train_coords = [train_coords[i] for i in keep]

    print(f"  Generating test conformers ({len(test_df)} molecules) ...")
    test_atoms, test_coords, test_valid = generate_conformers(
        test_df[SMILES_TEST].tolist(), remove_hs=REMOVE_HS, label="test")
    print(f"  Total conformer time: {(time.time()-t_conf)/60:.1f} min")

    # For test: use conformers only if all molecules succeeded
    use_test_conf = all(test_valid)
    if not use_test_conf:
        n_fail = sum(1 for v in test_valid if not v)
        print(f"  {n_fail} test conformers failed — "
              "test predictions will use SMILES only")
        test_atoms_arg  = None
        test_coords_arg = None
    else:
        test_atoms_arg  = test_atoms
        test_coords_arg = test_coords

    # ── Train all N_FOLDS folds in one call ───────────────────────────────────
    print(f"\n[Step 2/4] Training {N_FOLDS}-fold scaffold CV ...")
    train_smiles  = train_df['SMILES'].tolist()
    train_targets = train_df[TARGET_COL].values

    run_training(train_smiles, train_atoms, train_coords, train_targets,
                 KFOLD_SAVE_DIR)

    verify_saved_models(KFOLD_SAVE_DIR)
    elapsed = (time.time() - t0) / 60
    print(f"  Training done in {elapsed:.1f} min")

    # ── Test predictions (ensemble of all N_FOLDS models) ────────────────────
    print(f"\n[Step 3/4] Generating test predictions (ensemble of {N_FOLDS}) ...")
    flush_gpu()
    final_test_preds = run_inference(
        KFOLD_SAVE_DIR,
        test_df[SMILES_TEST].tolist(),
        test_atoms_arg,
        test_coords_arg,
    )
    print(f"  Test ensemble: "
          f"[{final_test_preds.min():.4f}, {final_test_preds.max():.4f}]  "
          f"mean={final_test_preds.mean():.4f}")

    # Save raw test predictions before any calibration
    raw_csv = OUTPUT_CSV.replace('.csv', '_raw.csv')
    save_predictions(test_df, final_test_preds, raw_csv)
    print("  *** Raw predictions saved — safe to proceed ***")

    # ── OOF reconstruction + isotonic calibration ─────────────────────────────
    if COMPUTE_OOF:
        print(f"\n[Step 4/4] OOF reconstruction ...")
        flush_gpu()
        oof_preds = compute_oof(
            KFOLD_SAVE_DIR,
            train_smiles, train_targets,
            train_atoms, train_coords,
        )

        if oof_preds is not None:
            oof_mae      = np.mean(np.abs(oof_preds - train_targets))
            oof_spearman = spearmanr(oof_preds, train_targets).correlation
            print(f"\n  Overall OOF MAE      : {oof_mae:.4f}")
            print(f"  Overall OOF Spearman : {oof_spearman:.4f}")

            # Save OOF predictions
            oof_df = train_df[['SMILES', TARGET_COL]].copy()
            oof_df['oof_pred'] = oof_preds
            oof_path = os.path.join(WORK_DIR, 'oof_predictions_unimol_20fold.csv')
            oof_df.to_csv(oof_path, index=False)
            print(f"  OOF saved → {oof_path}")

            if APPLY_CALIBRATION:
                print(f"\n  Fitting isotonic calibration on OOF ...")
                iso       = IsotonicRegression(out_of_bounds='clip',
                                               increasing=True)
                iso.fit(oof_preds, train_targets)

                cal_oof    = iso.predict(oof_preds)
                mae_before = np.mean(np.abs(oof_preds - train_targets))
                mae_after  = np.mean(np.abs(cal_oof   - train_targets))
                bias_bef   = np.mean(oof_preds - train_targets)
                bias_aft   = np.mean(cal_oof   - train_targets)
                print(f"  OOF MAE  : {mae_before:.4f} → {mae_after:.4f}  "
                      f"(Δ={mae_after - mae_before:+.4f})")
                print(f"  OOF bias : {bias_bef:+.4f} → {bias_aft:+.4f}")

                # Apply calibration to test predictions
                cal_test = iso.predict(final_test_preds)
                print(f"  Calibrated test: "
                      f"[{cal_test.min():.4f}, {cal_test.max():.4f}]  "
                      f"mean={cal_test.mean():.4f}")

                cal_csv = OUTPUT_CSV.replace('.csv', '_calibrated.csv')
                save_predictions(test_df, cal_test, cal_csv)
                print(f"  Using calibrated predictions as final output.")
                final_test_preds = cal_test
    else:
        print(f"\n[Step 4/4] OOF skipped (COMPUTE_OOF=False)")

    # ── Save final submission ─────────────────────────────────────────────────
    print(f"\n{sep}")
    print("Saving final submission ...")
    save_predictions(test_df, final_test_preds, OUTPUT_CSV)

    total_hours = (time.time() - t0) / 3600
    print(f"\n{sep}")
    print(f"Total elapsed: {total_hours:.2f} hours")

    if IS_KAGGLE:
        print("\nArchiving fold models ...")
        zip_base = os.path.join(WORK_DIR, 'unimol_kfold_20_models')
        shutil.make_archive(zip_base, 'zip', WORK_DIR,
                            os.path.basename(KFOLD_SAVE_DIR))
        size_gb = os.path.getsize(zip_base + '.zip') / 1e9
        print(f"  Archived: unimol_kfold_20_models.zip  ({size_gb:.2f} GB)")
        print("\nDownload from Kaggle Output tab:")
        print(f"  {os.path.basename(OUTPUT_CSV)}")
        if COMPUTE_OOF:
            print(f"  oof_predictions_unimol_20fold.csv")
        print(f"  unimol_kfold_20_models.zip")

    print(sep)
    restore_sleep()


if __name__ == '__main__':
    main()